

Consider one hundred modeling options for house price:

    House size, trying degrees 1 through 10
    Number of rooms, trying degrees 1 through 10
    Building Type

Hint: The dictionary of possible values that you make to give to GridSearchCV will have two elements instead of one.

Q1: Which model performed the best?

Q2: What downsides do you see of trying all possible model options? How might you go about choosing a smaller number of tuning values to try?


In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.compose import ColumnTransformer

In [ ]:
ames = pd.read_csv("/content/AmesHousing(1).csv")

In [ ]:
X = ames.drop("SalePrice", axis = 1) # use everything in ames other than saleprice bcoz saleprice is y
y = ames["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y)

In [ ]:
ct_poly = ColumnTransformer(
    [
        ('dummify', OneHotEncoder(sparse_output=False), ['Bldg Type']),
        ('polynomial', PolynomialFeatures(), ['Gr Liv Area', 'TotRms AbvGrd'])
    ]
)

pipeline_poly = Pipeline(
    [
        ('preprocessing', ct_poly),
        ('linear regression', LinearRegression())
    ]
).set_output(transform='pandas')

degrees = {'preprocessing__polynomial__degree': np.arange(1,11)}

grid_fitted = GridSearchCV(pipeline_poly, degrees, cv=5, scoring='r2')

grid_fitted.fit(X,y)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('dummify',
                                                                         OneHotEncoder(sparse_output=False),
                                                                         ['Bldg '
                                                                          'Type']),
                                                                        ('polynomial',
                                                                         PolynomialFeatures(),
                                                                         ['Gr '
                                                                          'Liv '
                                                                          'Area',
                                                                          'TotRms '
                                                                          'AbvGrd'])])),
                                       ('linear regression',
                                        LinearRegression())]),
             param_grid={'preprocessing__polynomial__degree': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])},
             scoring='r2')

In [ ]:
grid_fitted.cv_results_

{'mean_fit_time': array([0.02290063, 0.01583028, 0.01805158, 0.01781206, 0.01845016,
        0.01951604, 0.03875403, 0.03914809, 0.14569316, 0.11836119]),
 'std_fit_time': array([0.0137525 , 0.00023579, 0.00287508, 0.00115932, 0.00041487,
        0.00067137, 0.00423635, 0.01483773, 0.11963582, 0.13187423]),
 'mean_score_time': array([0.00949955, 0.00931954, 0.0094069 , 0.00960555, 0.01014409,
        0.01173782, 0.01982613, 0.01428084, 0.02516727, 0.02250204]),
 'std_score_time': array([5.97820560e-04, 2.63805003e-04, 4.89037364e-05, 1.28575194e-04,
        1.22680350e-04, 2.03021648e-03, 6.28140572e-03, 3.03568995e-03,
        1.13455829e-02, 7.28767150e-03]),
 'param_preprocessing__polynomial__degree': masked_array(data=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              mask=[False, False, False, False, False, False, False, False,
                    False, False],
        fill_value=999999),
 'params': [{'preprocessing__polynomial__degree': np.int64(1)},
  {'preprocessing__polynomial__d

In [ ]:
grid_fitted.cv_results_['mean_fit_time'].tolist()

[0.0229006290435791,
 0.015830278396606445,
 0.018051576614379884,
 0.017812061309814452,
 0.018450164794921876,
 0.01951603889465332,
 0.0387540340423584,
 0.03914809226989746,
 0.14569315910339356,
 0.11836118698120117]

## Q2: Downsides of trying all possible model options and how to choose a smaller number of tuning values.

Trying all possible model options, especially when dealing with multiple hyperparameters and a wide range of values for each, can have several downsides:

1.  **Computational Cost:** As the number of hyperparameters and their possible values increase, the number of combinations to evaluate grows exponentially. This can be computationally very expensive and time-consuming, requiring significant processing power and time.
2.  **Risk of Overfitting to the Validation Set:** While cross-validation helps to mitigate overfitting to the training data, trying an extremely large number of hyperparameter combinations can lead to overfitting to the validation set itself. The model might perform exceptionally well on the validation data but generalize poorly to new, unseen data.
3.  **Diminishing Returns:** Beyond a certain point, exploring an ever-increasing number of hyperparameter values may yield only marginal improvements in model performance, making the additional computational effort inefficient.
4.  **Difficulty in Interpretation:** With a vast number of models to compare, it can become challenging to understand why a particular combination of hyperparameters performs best and gain insights into the underlying relationships in the data.

**How to choose a smaller number of tuning values:**

Instead of exhaustively searching through every possible combination, you can employ more efficient strategies for hyperparameter tuning:

1.  **Domain Knowledge and Prior Experience:** If you have prior knowledge about the data or similar problems, you can use this to narrow down the range of hyperparameters to explore. Based on how different parameters typically affect model performance, you can make educated guesses about promising value ranges.
2.  **Random Search:** Instead of a grid search that checks every combination, random search samples a fixed number of random combinations from the specified hyperparameter distributions. This can be more efficient, especially when some hyperparameters have a greater impact on performance than others.
3.  **Bayesian Optimization:** More advanced techniques like Bayesian optimization build a probabilistic model of the objective function (e.g., cross-validation score) and use it to intelligently select the next hyperparameter combination to evaluate. This approach aims to find the optimal hyperparameters in fewer iterations than grid or random search.
4.  **Coarse-to-Fine Search:** Start with a wide range of values for each hyperparameter and a coarse grid. Once you identify promising regions, you can refine the search by focusing on a smaller range of values within those regions with a finer grid.
5.  **Visualize and Analyze Results:** After an initial search, visualize the performance across different hyperparameter values. This can help you identify trends and determine if the search space needs to be expanded or narrowed.

By using these strategies, you can significantly reduce the computational cost and time required for hyperparameter tuning while still having a high probability of finding good model configurations.